# NovaLake Gold Validation

This notebook validates Module 1 gold datasets after running the pipeline.

In [7]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()

In [8]:
GOLD_TABLES = [
    "daily_revenue",
    "sales_by_country",
    "top_products",
    "customer_revenue",
    "payment_success_rate",
    "shipment_delivery_summary",
]

In [9]:
spark.sql("SHOW TABLES IN novalake.gold").orderBy("tableName").show(truncate=False)

+---------+-------------------------+-----------+
|namespace|tableName                |isTemporary|
+---------+-------------------------+-----------+
|gold     |customer_revenue         |false      |
|gold     |daily_revenue            |false      |
|gold     |payment_success_rate     |false      |
|gold     |sales_by_country         |false      |
|gold     |shipment_delivery_summary|false      |
|gold     |top_products             |false      |
+---------+-------------------------+-----------+



In [10]:
counts = []
for table in GOLD_TABLES:
    n = spark.table(f"novalake.gold.{table}").count()
    counts.append((table, n))
spark.createDataFrame(counts, ["table_name", "row_count"]).orderBy("table_name").show(truncate=False)

+-------------------------+---------+
|table_name               |row_count|
+-------------------------+---------+
|customer_revenue         |950      |
|daily_revenue            |1155     |
|payment_success_rate     |4776     |
|sales_by_country         |6504     |
|shipment_delivery_summary|10551    |
|top_products             |100      |
+-------------------------+---------+



In [11]:
null_checks = [
    ("daily_revenue", ["order_date", "currency", "daily_revenue"]),
    ("sales_by_country", ["order_date", "country", "gross_revenue"]),
    ("top_products", ["product_id", "product_revenue", "units_sold"]),
    ("customer_revenue", ["customer_id", "lifetime_revenue"]),
    ("payment_success_rate", ["payment_date", "payment_method", "success_rate"]),
    ("shipment_delivery_summary", ["shipment_date", "shipping_country", "shipment_count"]),
]

for table, cols in null_checks:
    print(f"\n[NULL CHECK] {table}")
    df = spark.table(f"novalake.gold.{table}")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).show(truncate=False)


[NULL CHECK] daily_revenue
+----------+--------+-------------+
|order_date|currency|daily_revenue|
+----------+--------+-------------+
|0         |0       |0            |
+----------+--------+-------------+


[NULL CHECK] sales_by_country
+----------+-------+-------------+
|order_date|country|gross_revenue|
+----------+-------+-------------+
|0         |0      |0            |
+----------+-------+-------------+


[NULL CHECK] top_products
+----------+---------------+----------+
|product_id|product_revenue|units_sold|
+----------+---------------+----------+
|0         |0              |0         |
+----------+---------------+----------+


[NULL CHECK] customer_revenue
+-----------+----------------+
|customer_id|lifetime_revenue|
+-----------+----------------+
|0          |0               |
+-----------+----------------+


[NULL CHECK] payment_success_rate
+------------+--------------+------------+
|payment_date|payment_method|success_rate|
+------------+--------------+------------+
|0   

In [12]:
spark.sql("SELECT * FROM novalake.gold.daily_revenue ORDER BY order_date LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM novalake.gold.sales_by_country ORDER BY order_date, country LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM novalake.gold.top_products ORDER BY revenue_rank, product_id LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM novalake.gold.customer_revenue ORDER BY lifetime_revenue DESC LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM novalake.gold.payment_success_rate ORDER BY payment_date DESC, payment_method LIMIT 10").show(truncate=False)
spark.sql("SELECT * FROM novalake.gold.shipment_delivery_summary ORDER BY shipment_date DESC, shipping_country LIMIT 10").show(truncate=False)

+----------+--------+-------------+-----------+---------------+--------------------------+
|order_date|currency|daily_revenue|order_count|avg_order_value|_processed_timestamp      |
+----------+--------+-------------+-----------+---------------+--------------------------+
|2023-01-01|USD     |33696.72     |16         |2106.05        |2026-03-08 20:40:59.498117|
|2023-01-02|USD     |23375.84     |9          |2597.32        |2026-03-08 20:40:59.498117|
|2023-01-03|USD     |11109.32     |6          |1851.55        |2026-03-08 20:40:59.498117|
|2023-01-04|USD     |17572.98     |10         |1757.30        |2026-03-08 20:40:59.498117|
|2023-01-05|USD     |13501.57     |9          |1500.17        |2026-03-08 20:40:59.498117|
|2023-01-06|USD     |15644.08     |9          |1738.23        |2026-03-08 20:40:59.498117|
|2023-01-07|USD     |11408.08     |5          |2281.62        |2026-03-08 20:40:59.498117|
|2023-01-08|USD     |9381.25      |8          |1172.66        |2026-03-08 20:40:59.498117|